# 5. SiO₂ dielectric response, BECs, Hessian, and ionic dielectric

[Open this notebook in Colab](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/05_sio2_dielectric_hessian.ipynb)

This notebook loads its own bundled α-quartz model and structure, relaxes atomic positions at fixed cell and zero field, then evaluates the electronic dielectric tensor and BECs. It obtains the full force-constant Hessian by differentiating the MACEField model and combines it with the BECs for a harmonic ionic dielectric estimate.

The Hessian is calculated from the model; it is not present in the labeled SiO₂ dataset. The final values are model predictions for this relaxed structure, not held-out reference metrics.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def ensure_checkout(path, url):
    path = Path(path)
    if (path / ".git").is_dir():
        branch = subprocess.run(
            ["git", "-C", str(path), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch == "main":
            return
        subprocess.run(["git", "-C", str(path), "fetch", "origin", "main"], check=True)
        subprocess.run(
            ["git", "-C", str(path), "checkout", "-B", "main", "origin/main"],
            check=True,
        )
        return
    if path.exists() and any(path.iterdir()):
        raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", url, str(path)],
        check=True,
    )


if IN_COLAB:
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    PACKAGE_REPO = Path("/content/mace-field")
    ensure_checkout(TUTORIAL_REPO, TUTORIAL_URL)
    ensure_checkout(PACKAGE_REPO, PACKAGE_URL)
else:
    CWD = Path.cwd().resolve()
    TUTORIAL_REPO = next(
        (
            path
            for path in (CWD, *CWD.parents)
            if (path / "tutorials/macefield/data").is_dir()
            and (path / "README.md").is_file()
        ),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run this notebook from a clone of mdi-group/mace-field-tutorial, "
            "or open it in Colab."
        )

    requested_repo = os.environ.get("MACEFIELD_REPO")
    if requested_repo:
        PACKAGE_REPO = Path(requested_repo).expanduser().resolve()
        if not (PACKAGE_REPO / "mace/modules/extensions.py").is_file():
            raise RuntimeError("MACEFIELD_REPO does not point to a MACE-Field checkout.")
        branch = subprocess.run(
            ["git", "-C", str(PACKAGE_REPO), "branch", "--show-current"],
            capture_output=True,
            text=True,
            check=False,
        ).stdout.strip()
        if branch != "main":
            raise RuntimeError("Set MACEFIELD_REPO to a checkout on the main branch.")
    else:
        candidates = [
            TUTORIAL_REPO.parent / "mace-field-main",
            TUTORIAL_REPO.parent / "mace-field",
        ]
        PACKAGE_REPO = None
        for candidate in candidates:
            if not (candidate / "mace/modules/extensions.py").is_file():
                continue
            branch = subprocess.run(
                ["git", "-C", str(candidate), "branch", "--show-current"],
                capture_output=True,
                text=True,
                check=False,
            ).stdout.strip()
            if branch == "main":
                PACKAGE_REPO = candidate.resolve()
                break
        if PACKAGE_REPO is None:
            PACKAGE_REPO = Path("/tmp/mace-field-main")
            ensure_checkout(PACKAGE_REPO, PACKAGE_URL)

os.environ["MACEFIELD_REPO"] = str(PACKAGE_REPO)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
sys.path.insert(0, str(PACKAGE_REPO))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(PACKAGE_REPO)],
    check=True,
)
os.chdir(TUTORIAL_REPO)

TUTORIAL_DIR = TUTORIAL_REPO / "tutorials/macefield"
DATA_DIR = TUTORIAL_DIR / "data"
print("Tutorial repository:", TUTORIAL_REPO)
print("MACE-Field source:", PACKAGE_REPO)


In [ ]:
import numpy as np
import torch
from ase.io import read
from ase.optimize import BFGS
from mace.calculators import MACECalculator
from mace.tools import torch_tools

MODEL = DATA_DIR / "sio2/MACE-field-SiO2.model"
SIO2_DATA = DATA_DIR / "sio2/SiO2-preprocessed.extxyz"
atoms = read(SIO2_DATA, index=0)
atoms.info.clear()
for key in list(atoms.arrays):
    if key not in {"numbers", "positions"}:
        atoms.arrays.pop(key)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
relax_calc = MACECalculator(
    model_paths=str(MODEL),
    model_type="MACEField",
    device=DEVICE,
    default_dtype="float64",
    electric_field=[0.0, 0.0, 0.0],
    compute_polarization=False,
    compute_becs=False,
    compute_polarizability=False,
)
atoms.calc = relax_calc
BFGS(atoms, logfile=None).run(fmax=0.05, steps=60)
print("atoms:", len(atoms), "| volume (Å³):", atoms.get_volume())
print("maximum force after fixed-cell relaxation (eV/Å):",
      float(np.linalg.norm(atoms.get_forces(), axis=1).max()))


## Evaluate electronic susceptibility and BECs

The calculator returns polarization in e/Å², BECs in e, and the volume-normalized electronic susceptibility tensor. Add the identity tensor to susceptibility to obtain $\epsilon_\infty$.


In [ ]:
response_calc = MACECalculator(
    model_paths=str(MODEL),
    model_type="MACEField",
    device=DEVICE,
    default_dtype="float64",
    electric_field=[0.0, 0.0, 0.0],
    compute_polarization=True,
    compute_becs=True,
    compute_polarizability=True,
)
atoms.calc = response_calc
atoms.get_potential_energy()
polarization = np.asarray(response_calc.results["polarization"], dtype=float)
becs = np.asarray(response_calc.results["becs"], dtype=float).reshape(len(atoms), 3, 3)
chi_electronic = np.asarray(
    response_calc.results["polarizability"], dtype=float
).reshape(3, 3)
epsilon_infinity = np.eye(3) + chi_electronic

print("polarization (e/Å²):", polarization)
print("BEC tensor shape:", becs.shape)
print("mean BEC over atoms (e):")
print(becs.mean(axis=0))
print("electronic susceptibility:")
print(chi_electronic)
print("epsilon_infinity:")
print(epsilon_infinity)


## Differentiate the field-conditioned energy twice with respect to positions

The public ASE Hessian helper currently accepts ordinary MACE models. MACEField's forward method already supports the same exact autograd Hessian, so the small helper below builds the graph through the calculator and calls that model method directly. The result is the fixed-cell Cartesian Hessian in eV/Å².


In [ ]:
def macefield_cartesian_hessian(atoms, calculator):
    batch = calculator._clone_batch(calculator._atoms_to_batch(atoms))
    batch_dict = batch.to_dict()
    model = calculator.models[0]
    model_dtype = next(model.parameters()).dtype
    for key in batch.keys:
        value = batch[key]
        if torch.is_tensor(value) and torch.is_floating_point(value):
            batch_dict[key] = value.to(dtype=model_dtype)

    field = calculator._resolve_electric_field(atoms).to(
        device=batch_dict["positions"].device,
        dtype=batch_dict["positions"].dtype,
    )
    batch_dict["electric_field"] = field

    with torch.enable_grad(), torch_tools.default_dtype(calculator.default_dtype):
        output = model(
            batch_dict,
            compute_force=True,
            compute_hessian=True,
            compute_stress=False,
            training=False,
            compute_polarization=False,
            compute_becs=False,
            compute_polarizability=False,
            electric_field=field,
        )
    hessian = output["hessian"].detach().cpu().numpy().reshape(3 * len(atoms), -1)
    return 0.5 * (hessian + hessian.T)

hessian = macefield_cartesian_hessian(atoms, relax_calc)
print("Hessian shape:", hessian.shape, "| units: eV/Å²")
print("maximum asymmetry after symmetrization:",
      float(np.max(np.abs(hessian - hessian.T))))


## Combine BECs and Hessian for the harmonic ionic response

At fixed cell, the static harmonic displacement response is $\chi_\mathrm{ion}=Z H^+ Z^T/(\Omega\epsilon_0)$, where $H^+$ is the pseudoinverse after removing translational zero modes. Here $\epsilon_0$ is expressed in the matching e/(V Å) unit convention. The calculation excludes negative and near-zero Hessian eigenvalues; any additional negative modes indicate an unstable structure and make the harmonic dielectric estimate unreliable.


In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(hessian)
cutoff = max(float(np.max(np.abs(eigenvalues))) * 1.0e-8, 1.0e-8)
stable_modes = eigenvalues > cutoff
negative_modes = int(np.count_nonzero(eigenvalues < -cutoff))
hessian_pinv = (
    eigenvectors[:, stable_modes] / eigenvalues[stable_modes]
) @ eigenvectors[:, stable_modes].T

# Z[polarization direction, Cartesian displacement coordinate]
z_matrix = np.transpose(becs, (1, 0, 2)).reshape(3, -1)
eps0_e_per_v_angstrom = 8.8541878128e-12 / 1.602176634e-19 / 1.0e10
chi_ionic = (
    z_matrix @ hessian_pinv @ z_matrix.T
    / (atoms.get_volume() * eps0_e_per_v_angstrom)
)
epsilon_static_harmonic = epsilon_infinity + chi_ionic

print("Hessian eigenvalues below the acoustic cutoff:", negative_modes)
print("retained positive modes:", int(np.count_nonzero(stable_modes)))
print("ionic susceptibility:")
print(chi_ionic)
print("harmonic static dielectric tensor:")
print(epsilon_static_harmonic)
if negative_modes:
    print("Warning: the relaxed model structure has unstable modes; do not interpret this harmonic ionic estimate as a stable dielectric constant.")
